In [1]:
import pandas as pd
import yfinance as yf
import pyupbit
import warnings
from datetime import datetime,timedelta
warnings.filterwarnings('ignore')

#시작 기준 날짜 설정
end_date_obj=datetime.today()
start_date_obj=end_date_obj-timedelta(days=10)

# yfinance는 end_date를 포함하지 않으므로 하루 더함.
yf_end_date = (end_date_obj + timedelta(days=1)).strftime('%Y-%m-%d')
start_date = start_date_obj.strftime('%Y-%m-%d')


print(f"분석 기간: {start_date} ~ {end_date_obj.strftime('%Y-%m-%d')}\n")


# 티커 매핑 (현금성 자산은 None으로 설정하여 변동성 0% 처리)
ticker_map_upbit = {'BTC': 'KRW-BTC', 'ETH': 'KRW-ETH'}
ticker_map_yf = {'VOO': 'VOO', 'QQQ': 'QQQ'}

df_init=pd.read_csv("/Users/woo/Desktop/weekly_stock_report/part/portfolio_rebalancing/portfolio_for_tableau.csv")
current_total_capital=df_init['Total_Value_KRW'].sum()
target_weights = {row['Asset']: row['Target_Weight'] / 100.0 for _, row in df_init.iterrows()}

# 1) 업비트 데이터 (KRW 기준)
upbit_prices = pd.DataFrame()
for asset, ticker in ticker_map_upbit.items():
    if asset in target_weights or asset == 'BTC':
        df = pyupbit.get_ohlcv(ticker, interval="day", to=end_date_obj, count=10)
        if df is not None:
            upbit_prices[asset] = df['close']

# 타임존 제거 (KST 날리기)
if not upbit_prices.empty:
    if upbit_prices.index.tz is not None:
        upbit_prices.index = upbit_prices.index.tz_localize(None)
    upbit_prices.index = upbit_prices.index.normalize()

# 2) 야후 파이낸스 데이터 (USD 기준)
yf_tickers = [ticker_map_yf[a] for a in target_weights.keys() if a in ticker_map_yf]
yf_df = pd.DataFrame()

if len(yf_tickers) > 0:
    yf_raw = yf.download(yf_tickers, start=start_date, end=yf_end_date)['Close']
    if isinstance(yf_raw, pd.Series):
        yf_df = yf_raw.to_frame(name=yf_tickers[0])
    else:
        yf_df = yf_raw

    # 타임존 완벽 제거 (EST 날리기)
    if not yf_df.empty:
        if yf_df.index.tz is not None:
            yf_df.index = yf_df.index.tz_localize(None)
        yf_df.index = yf_df.index.normalize()

# 3) 정제된 날짜 기준으로 병합 (Concat)
price_data = pd.concat([upbit_prices, yf_df], axis=1)

# 미국 주식 주말 빈칸(NaN)을 금요일 종가로 채우기 (Forward Fill)
price_data = price_data.ffill().bfill()

price_data = price_data.tail(8)
returns_data = price_data.pct_change().fillna(0)

#주간 누적 수익률: 3가지 경우(7일전 평가금액과 오늘 평가금 비교)
asset_weekly_returns=(price_data.iloc[-1]/price_data.iloc[0])-1

def get_return(asset_name):
    if asset_name in asset_weekly_returns:
        return asset_weekly_returns[asset_name]
    return 0.0

#내 포트폴리오 수익률
my_weekly_return=sum([get_return(asset) * weight for asset, weight in target_weights.items()])

#비트코인 100% 투자 후 홀딩
bm1_weekly_return=get_return('BTC')

#비트코인 50%+VOO 50%
bm2_weekly_return=(get_return('BTC') * 0.5) + (get_return('VOO') * 0.5)


# 태블로 시각화용 데이터프레임 생성 및 저장
report_data = {
    'Strategy': ['1. Benchmark (100% BTC)', '2. Benchmark (50% BTC + 50% VOO)', '3. My Portfolio (Shannon)'],
    'Weekly_Return_Pct': [bm1_weekly_return * 100, bm2_weekly_return * 100, my_weekly_return * 100]
}

df_report = pd.DataFrame(report_data)

# 기존과 동일한 폴더에 주간 요약용 CSV로 저장
save_path = "/Users/woo/Desktop/weekly_stock_report/part/portfolio_rebalancing/weekly_return_summary.csv"
df_report.to_csv(save_path, index=False)

#1.my_portfolio를 실행하면 실행한 시간 기준의 데이터가 새롭게 갱신된다.
#2.이 코드가 portfolio_for_taleau.csv 파일을 가지고 설정한 비중, 비트코인 100%, 비트코인 voo50%씩 했을때 이 코드를 돌린 날짜 기준으로 7일 전과 지금 시장가를 비교해서 주간 수익률을 계산



분석 기간: 2026-04-16 ~ 2026-04-26



[*********************100%***********************]  1 of 1 completed
